# Using the DynamicCallable, DynamicFunction, and DynamicMethod Classes

## Introduction

The DynamicCallable family provides callable wrapper classes with multiplexed binding and callback. They integrate MethodMultiplexer under the hood to dynamically choose how binding and calling occur.

This tutorial focuses on DynamicCallable and highlights the method multiplexer functionality for binding and callback. Separate module tutorials for DynamicFunction and DynamicMethod expand on specifics.

### What you will learn
- Core concepts of DynamicCallable
- How bind and call multiplexers work (MethodMultiplexer)
- Changing bind_method and call_method at runtime
- Working with functions vs methods
- Practical examples and tips

### Table of Contents
- Importing
- Core concepts
- Binding multiplexer
- Calling multiplexer
- Examples
- FAQs


## Importing

In [ ]:
from baseobjects.functions import DynamicCallable, DynamicFunction, DynamicMethod
from baseobjects.functions import MethodMultiplexer
from baseobjects.functions import CallableMultiplexer


## Core Concepts

DynamicCallable is an abstract base for callable wrappers with two internal MethodMultiplexer instances:
- bind_multiplexer: controls how the object binds when used as a descriptor (__get__).
- call_multiplexer: controls how the object dispatches when called (__call__).

Both multiplexers can select different strategies (methods) by name. Properties bind_method and call_method are convenience accessors that also update defaults.


## Binding Multiplexer

When a DynamicCallable is used as a descriptor on a class, __get__ delegates to bind_multiplexer. The selected binding strategy determines what object is returned (e.g., a bound method wrapper, the instance itself, etc.). Common binding methods include:
- bind_builtin: default for DynamicCallable (acts like normal functions/methods)
- bind_self: default for DynamicMethod

You can switch binding strategy at runtime:

In [ ]:
class BinderDemo(DynamicCallable):
    pass

b = BinderDemo()
print('Default bind method:', b.bind_method)

# Change at runtime
b.bind_method = 'bind_builtin'
print('Changed bind method:', b.bind_method)


## Calling Multiplexer

The __call__ of DynamicCallable delegates to call_multiplexer. The selected call method controls how the wrapped function is invoked. Common call methods include:
- call_wrapped: call the underlying function as-is
- other project-specific call strategies if registered

You can switch call strategy at runtime:

In [ ]:
class CallerDemo(DynamicCallable):
    pass

c = CallerDemo()
print('Default call method:', c.call_method)

c.call_method = 'call_wrapped'
print('Changed call method:', c.call_method)


## Examples

### Example 1: Basic wrapper that forwards to a function

In [ ]:
from baseobjects.functions import FunctionRegistry

class Adder(DynamicCallable):
    def __init__(self, func=None, **kwargs):
        super().__init__(func=func, **kwargs)

# A simple function
def add(a, b):
    return a + b

adder = Adder(func=add)
print('add(2,3)=', adder(2,3))

# Switch call strategy (still using call_wrapped by default)
adder.call_method = 'call_wrapped'
print('after switch add(4,5)=', adder(4,5))


### Example 2: Descriptor binding behavior

In [ ]:
class Greeter:
    def __init__(self, name):
        self.name = name

class GreeterCallable(DynamicCallable):
    def __init__(self, func=None, **kwargs):
        super().__init__(func=func, **kwargs)

    def call_wrapped(self, *args, **kwargs):
        # demonstrate that this call path is selectable
        return self._wrapped_(*args, **kwargs)

# A function that expects an instance and a message
def greet(self, msg):
    return f"{self.name}: {msg}"

Greeter.speak = GreeterCallable(func=greet)

bob = Greeter('Bob')
# __get__ uses bind_multiplexer; default_bind_method is bind_builtin on DynamicCallable
print(bob.speak('Hello'))

# If needed, choose a different binding strategy
Greeter.speak.bind_method = 'bind_builtin'
print(bob.speak('Hi again'))


## FAQs

Q: How are strategies registered?\n
A: DynamicCallable uses MethodMultiplexer which can select methods defined on the instance (like call_wrapped) or in an attached registry. See methodmultiplexer_tutorial for adding/selecting functions.

Q: What differs among DynamicCallable, DynamicFunction, DynamicMethod?\n
A: DynamicFunction is tailored for wrapping free functions; DynamicMethod adapts to bound instance methods and overrides __call__ to automatically pass self._self_(). See their dedicated tutorials.
